In [1]:
#------------------------------------------------ Import Lib ----------------------------------------
import os
import re
import json
import shutil
import datetime
import requests
import pandas as pd
import pdfplumber

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'RW NBRW' ## National Bank of Rwanda

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd() ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running RW NBRW Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
          'Phone - Mother company': []}

# Source pages from Jira DECD-6292 (kept for reference / traceability)
regdict={
        regulatorName + ' 1': 'https://www.bnr.rw/banksupervision',
        regulatorName + ' 2': 'https://www.bnr.rw/microfinance',
        regulatorName + ' 4': 'https://www.bnr.rw/insurances',
        regulatorName + ' 5': 'https://www.bnr.rw/pensions',
        regulatorName + ' 6': 'https://www.bnr.rw/paymentsystem',
        regulatorName + ' 7': 'https://www.bnr.rw/insurances',
        }

Typology={
       regulatorName + ' 1': 'List of Licensed Banks',
       regulatorName + ' 2': 'List of MFIs',
       regulatorName + ' 4': 'List of Licensed Insurance and Reinsurance Brokers',
       regulatorName + ' 5': 'List of Pension service providers',
       regulatorName + ' 6': 'List of Payment System institutions',
       regulatorName + ' 7': 'List of Insurance Companies',
        }

# ListLabel = 1 for bank lists, 2 for insurance, 3 for bank & insurance, 4 for everything else
listlabeldict={
       regulatorName + ' 1': '1',
       regulatorName + ' 2': '1',
       regulatorName + ' 4': '2',
       regulatorName + ' 5': '4',
       regulatorName + ' 6': '4',
       regulatorName + ' 7': '2',
        }

# The bnr.rw pages are a React SPA: each "chart of documents" is served by a JSON endpoint.
# Documents are matched by name (not by file path) because the pdf filenames change with
# every monthly re-upload, e.g. "...-April_2026_DG6v9R9.pdf".
apidict={
       regulatorName + ' 1': ('https://www.bnr.rw/fsbs',   'List of Supervised Banks'),
       regulatorName + ' 2': ('https://www.bnr.rw/fsmf',   'List of Supervised Deposit-taking Microfinance Institutions'),
       regulatorName + ' 4': ('https://www.bnr.rw/fsins',  'List of Licensed Insurance and Reinsurance Brokers'),
       regulatorName + ' 5': ('https://www.bnr.rw/fspens', 'List of Pension service providers'),
       regulatorName + ' 6': ('https://www.bnr.rw/fsps',   'Licensed Institutions as of'),
       regulatorName + ' 7': ('https://www.bnr.rw/fsins',  'List of Insurance Companies'),
        }

processdate = now.strftime('%Y-%m-%d')

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'}

In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")


def clean_text(s):
    if not s:
        return ''
    return re.sub(r'\s+', ' ', str(s)).strip()


def clean_email(s):
    # pdf text layer inserts stray spaces, e.g. "rwanda@asa -international.com"
    return re.sub(r'\s+', '', s or '').strip()


def clean_phone(s):
    s = clean_text(s)
    s = re.sub(r'(?<=\d)\s+(?=\d)', '', s)   # "7 88642568" -> "788642568"
    s = re.sub(r'\s*([-/])\s*', r'\1', s)    # "504239 -504240" -> "504239-504240"
    s = re.sub(r'\(\s*\+', '(+', s)
    return s.strip(' ,;.')


def download_list_pdf(reg):
    """Resolve the target document on the bnr.rw JSON endpoint by name and download it."""
    endpoint, docname = apidict[reg]
    r = requests.get(endpoint, headers=HEADERS, timeout=60, verify=False)
    r.raise_for_status()
    docs = [d for d in r.json() if d.get('name', '').strip().lower().startswith(docname.lower())]
    if not docs:
        raise ValueError(f'No document matching {docname!r} on {endpoint}')
    docs.sort(key=lambda d: d.get('date_last_modified', ''), reverse=True)  # newest re-upload wins
    doc = docs[0]
    url = 'https://www.bnr.rw' + doc['file']
    localpath = os.path.join(tempfolder, reg.replace(' ', '_') + '_' + os.path.basename(doc['file']))
    fr = requests.get(url, headers=HEADERS, timeout=120, verify=False)
    fr.raise_for_status()
    with open(localpath, 'wb') as f:
        f.write(fr.content)
    print(f"   downloaded {doc['name']!r} -> {os.path.basename(localpath)}")
    return localpath


def extract_table_rows_text(pdf_path):
    """All table rows of a text-layer pdf, as lists of cell strings (pdfplumber)."""
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    rows.append([c if c is not None else '' for c in row])
    return rows


def extract_rows_list6(pdf_path):
    """PSP table rows, plus the two bullet-list sections that follow the table on the
    same pdf (Payment System Operators / Cheque printing-encoding companies). Those are
    plain bulleted paragraphs, not gridded cells, so pdfplumber's extract_tables() never
    picks them up - pull them from the page text instead and tag them as ('BULLET', name,
    section) rows so parse_list6_payment can tell them apart from the table rows."""
    rows = extract_table_rows_text(pdf_path)
    section_map = {
        'payment system operators': 'Payment System Operator',
        'cheque printing/encoding companies': 'Cheque Printing/Encoding Company',
    }
    section = None
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for line in (page.extract_text() or '').split('\n'):
                line = line.strip()
                if line.lower() in section_map:
                    section = section_map[line.lower()]
                    continue
                m = re.match(r'^[•\-]\s*(.+)', line)
                if m and section:
                    # pdf text layer inserts stray spaces around hyphens, e.g. "FZ -LLC"
                    name = re.sub(r'(?<=[A-Za-z])\s+-(?=[A-Za-z])', '-', clean_text(m.group(1)))
                    rows.append(['BULLET', name, section])
    return rows


def extract_table_rows_ocr(pdf_path, borderless_first=False):
    """All table rows of an image-based pdf, as lists of cell strings.
    Uses img2table + Tesseract (production convention; Windows box has Tesseract-OCR).
    pdf_text_extraction=False is required: these pdfs carry a couple of real text-layer
    words (page titles / column headers) alongside data cells rendered as vector glyphs
    with no extractable text. img2table's default (pdf_text_extraction=True) treats any
    non-empty text layer as authoritative and skips Tesseract entirely, so every data cell
    comes back blank. Forcing it off makes img2table always OCR the rendered page image."""
    from img2table.document import PDF as Img2TablePDF
    if os.name == 'nt':
        tess_dir = r"C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\Tesseract-OCR"
        os.environ["PATH"] = tess_dir + ";" + os.environ.get("PATH", "")

    if shutil.which('tesseract'):
        from img2table.ocr import TesseractOCR
        ocr = TesseractOCR(n_threads=1, lang="eng")
    else:
        # No tesseract binary on this machine (e.g. the Mac, no brew/admin access) -
        # fall back to img2table's pure-Python RapidOCR engine. Keep Tesseract as the
        # preferred engine (better word spacing on tightly-kerned all-caps names); this
        # fallback is an unblock, so flag Mac-run output for QA rather than trusting it
        # as the authoritative run.
        from img2table.ocr import RapidOCR
        ocr = RapidOCR()

    try:
        from importlib.metadata import version as _pkg_version
        i2t_ver = _pkg_version('img2table')
    except Exception:
        i2t_ver = '?'
    print(f'   [OCR] engine={type(ocr).__name__}, img2table={i2t_ver}')

    doc = Img2TablePDF(pdf_path, pdf_text_extraction=False)

    def run(borderless):
        extracted = doc.extract_tables(ocr=ocr, implicit_rows=False,
                                       borderless_tables=borderless, min_confidence=50)
        rows = []
        for page_tables in extracted.values():
            for table in page_tables:
                for row in table.content.values():
                    rows.append([cell.value if cell.value else '' for cell in row])
        return rows

    rows = run(borderless_first)
    print(f'   [OCR] borderless_tables={borderless_first}: {len(rows)} raw rows')
    if len(rows) < 5:
        # A thin result means the detector latched onto a partial grid fragment (or
        # found nothing at all). Don't key the retry on 0 rows only: on the production
        # box the bordered pass returned a small non-empty fragment for the pension
        # pdf, which blocked the borderless retry and shipped just 3 of 14 entities.
        alt = run(not borderless_first)
        print(f'   [OCR] retry borderless_tables={not borderless_first}: {len(alt)} raw rows')
        if len(alt) > len(rows):
            rows = alt
    for r in rows[:40]:
        print('   [OCR]    ' + ' | '.join(c[:28] for c in r))
    return rows


def extract_table_rows_ocr_borderless(pdf_path):
    """List 5 (pension): the grid is drawn as filled rects with zero stroked lines, so
    the borderless detector is the reliable one - run it first instead of hoping the
    bordered pass returns exactly 0 rows and falls through."""
    return extract_table_rows_ocr(pdf_path, borderless_first=True)


def nonempty(cells):
    return [clean_text(c) for c in cells if clean_text(c)]

In [5]:
#------------------------------------------------ Begin_Parsers ----------------------------------------
def parse_list1_banks(rows, reg):
    """No | Bank Name | Category -- Category is a vertically merged cell, fill it down.
    OCR of the tiny single-digit "No" cells is unreliable, so rows are identified by
    their Bank Name instead of the running number."""
    skip_re = re.compile(r'^no\.?$|bank name|category|list of supervised|bnr restricted', re.I)
    last_category = ''
    for row in rows:
        cells = [c for c in nonempty(row) if not (c.isdigit() and len(c) <= 2)]  # drop running number
        if not cells or skip_re.search(cells[0]):
            continue  # header / title rows
        name = cells[0]
        category = cells[1] if len(cells) > 1 else ''
        if category:
            last_category = category
        sqldict['Name'].append(name)
        sqldict['License_Type'].append(category if category else last_category)
        add_common(reg)


def parse_list2_mfi(rows, reg):
    """# | Name | Location (District) | Location (Province) | Email | Category"""
    for row in rows:
        cells = nonempty(row)
        if len(cells) < 6 or not cells[0].isdigit():
            continue
        name, district, province, email, category = cells[1], cells[2], cells[3], cells[4], cells[5]
        sqldict['Name'].append(name)
        sqldict['City'].append(district.title())
        sqldict['Address_2'].append(province.title())
        sqldict['Email'].append(clean_email(email))
        sqldict['License_Type'].append(re.sub(r'\s*-\s*', '-', category))  # "Deposit -Taking" -> "Deposit-Taking"
        add_common(reg)


def parse_list4_brokers(rows, reg):
    """S/N | Insurance Broker | License Number | Telephone | Category"""
    for row in rows:
        cells = nonempty(row)
        if len(cells) < 5 or not cells[0].isdigit():
            continue
        name, license_no, phone, category = cells[1], cells[2], cells[3], cells[4]
        sqldict['Name'].append(name)
        if license_no.upper() != 'NA':
            sqldict['InternalID_1'].append(license_no)
            sqldict['InternalID_1_type'].append('License Number')
        sqldict['Phone'].append(clean_phone(phone))
        sqldict['License_Type'].append(category)
        add_common(reg)


def parse_list5_pension(rows, reg):
    """Sections (CORPORATE TRUSTEE / ADMINISTRATORS / INVESTMENT MANAGERS / CUSTODIANS),
    then # | Name | License Number/Accreditation Letter Ref. | Contacts.
    Rows are identified by name (the tiny "#" cells OCR unreliably); remaining cells are
    classified as contact vs license by shape, so a missing license ("-") can't shift columns."""
    # OCR sometimes concatenates the header with no space ("CORPORATETRUSTEE",
    # "INVESTMENTMANAGERS") - \s* tolerates that, and mapping to a canonical name (rather
    # than title-casing the raw match) keeps License_Type as "Corporate Trustee" /
    # "Investment Managers" instead of "Corporatetrustee" / "Investmentmanagers".
    section_patterns = [
        (re.compile(r'CORPORATE\s*TRUSTEE', re.I), 'Corporate Trustee'),
        (re.compile(r'ADMINISTRATORS', re.I), 'Administrators'),
        (re.compile(r'INVESTMENT\s*MANAGERS', re.I), 'Investment Managers'),
        (re.compile(r'CUSTODIANS', re.I), 'Custodians'),
    ]
    header_re = re.compile(r'type of service providers|list of licensed|^license|^contacts', re.I)
    section = ''
    for row in rows:
        cells = nonempty(row)
        if not cells:
            continue
        joined = ' '.join(cells)
        matched_section = next((canon for pat, canon in section_patterns if pat.search(joined)), None)
        if matched_section:
            section = matched_section
            continue
        while cells and len(cells[0]) <= 2:  # drop the running-number cell (or OCR junk in it)
            cells.pop(0)
        if not cells or header_re.search(cells[0]):
            continue
        name = cells[0]
        license_no = contact = ''
        for c in cells[1:]:
            flatc = c.replace(' ', '')
            if '@' in c or re.match(r'^\(?\+', flatc) or re.fullmatch(r'[\d\s()+-]+', c):
                contact = c
            elif c not in ('-', '--'):
                license_no = c
        sqldict['Name'].append(name)
        if license_no:
            sqldict['InternalID_1'].append(clean_text(license_no))
            sqldict['InternalID_1_type'].append('License/Accreditation Letter Number')
        emails = EMAIL_RE.findall(contact.replace(' ', ''))
        if emails:
            sqldict['Email'].append(emails[0])
        elif contact:
            sqldict['Phone'].append(clean_phone(re.sub(r'Email\s*:?', '', contact, flags=re.I)))
        sqldict['License_Type'].append(section)
        add_common(reg)


def parse_list6_payment(rows, reg):
    """No | PSP Name | E-Money Issuer | Aggregator | Remittance (check marks)"""
    services = ['E-Money Issuer', 'Aggregator', 'Remittance']
    for row in rows:
        if len(row) < 5:
            continue  # title block table
        no, name = clean_text(row[0]), clean_text(row[1])
        if not no.isdigit() or not name:
            continue
        licensed_for = [svc for svc, cell in zip(services, row[2:5]) if clean_text(cell)]
        sqldict['Name'].append(name)
        sqldict['License_Type'].append(', '.join(licensed_for))
        add_common(reg)


def parse_list7_insurers(rows, reg):
    """Sections (PUBLIC / PRIVATE / MICRO / CAPTIVE INSURERS, HMOs, Mutual Insurers);
    columns: # | Name | details blob (address/phone/website/email) | Insurance Category.
    A continuation row (no # / no name) carries leftover details of the previous insurer."""
    section_re = re.compile(r'(PUBLIC|PRIVATE|MICRO|CAPTIVE)\s+INSURERS|Health\s+M\s?edical\s+Organizations|Mutual\s+Insurers', re.I)
    detail_keyword_re = re.compile(r'licensed in|telephone|tel\s*[:/]|fax|e-?\s?mail|web\s*-?\s*site|website|established by|law n|determining|organization and functioning|currently located', re.I)
    section = ''

    def parse_details(detail):
        # phones/emails/websites wrap across pdf lines, so extract from flattened text;
        # addresses need the line structure, so those come from the raw lines below
        flat = clean_text(detail)
        phone = fax = website = email = ''
        m = EMAIL_RE.search(flat)
        if m:
            email = m.group(0)
        m = re.search(r'Fax\s*:?\s*([+()\d\s/-]{6,})', flat, re.I)
        if m:
            fax = clean_phone(m.group(1))
        m = re.search(r'Tel(?:ephone)?(?:\s*/\s*Fax)?\s*:?\s*(.+)', flat, re.I)
        if m:
            captured = re.sub(r'(Fax|E-?\s?mail|Web)\s*-?\s*.*$', '', m.group(1), flags=re.I)
            # keep only phone-shaped tokens: skips prose like carrier names or legal text
            tokens = [t for t in re.findall(r'[+(]{0,2}\d[\d\s()/-]*\d', captured)
                      if len(re.sub(r'\D', '', t)) >= 7]
            phone = ' / '.join(clean_phone(t) for t in tokens)
        m = re.search(r'Web\s*-?\s*site\s*:?', flat, re.I)
        if m:
            chunk = re.sub(r'\s+', '', flat[m.end():m.end() + 80])
            # websites in these pdfs are lowercase; a case-sensitive match stops at
            # trailing OCR debris like "www.mmi.gov.rw KIGALI-RWANDA"
            mm = re.match(r'(https?://[a-z0-9./_-]+|www\.[a-z0-9.-]+)', chunk)
            if mm:
                website = mm.group(1).rstrip('.')
        addr_lines = []
        for l in detail.split('\n'):
            l = l.strip()
            km = detail_keyword_re.search(l)
            if km:  # keep any address text before the keyword, e.g. "P.O. Box 175 ... Tel: ..."
                l = l[:km.start()].strip(' ,;.')
            if not l or EMAIL_RE.search(l):
                continue
            if re.fullmatch(r'[\d\s()+/,.;-]+', l):    # phone/fax continuation line
                continue
            if re.fullmatch(r'(https?://)?[a-z0-9.-]+\.[a-z]{2,}[/a-z0-9.-]*', l):  # bare domain
                continue
            if len(l) <= 12 and not re.search(r'\d', l):  # orphan wrap fragment, e.g. "business)"
                continue
            addr_lines.append(l)
        address = clean_text(', '.join(addr_lines))
        return address, phone, fax, website, email

    for row in rows:
        cells_all = [clean_text(c) for c in row]
        joined = ' '.join([c for c in cells_all if c])
        m = section_re.search(joined)
        if m and not cells_all[0].isdigit():
            section = re.sub(r'M\s+edical', 'Medical', clean_text(m.group(0)), flags=re.I).title()
            continue
        raw_cells = [c if c is not None else '' for c in row]
        no = cells_all[0]
        named = [c for c in cells_all[1:] if c]
        if not no.isdigit():
            # continuation row: patch previous record with any extra details found
            detail = '\n'.join([c for c in raw_cells if clean_text(c)])
            if detail and sqldict['Name']:
                address, phone, fax, website, email = parse_details(detail)
                for field, value in [('Phone', phone), ('Fax', fax), ('Website', website), ('Email', email)]:
                    if value and not sqldict[field][-1]:
                        sqldict[field][-1] = value
            continue
        if len(named) < 2:
            continue
        name = named[0]
        detail_raw = ''
        for c in raw_cells[2:]:
            if clean_text(c) and clean_text(c) != name and len(clean_text(c)) > len(detail_raw):
                detail_raw = c
        category = named[-1] if len(named) > 2 else ''
        address, phone, fax, website, email = parse_details(detail_raw)
        category = re.sub(r'Lo\s?ng', 'Long', category)       # "Lo ng Term" pdf artifact
        category = re.sub(r'\s+-\s+', '-', category)          # "Long - Term" -> "Long-Term"
        category = re.sub(r'\s+\)', ')', category)            # "Insurance )" -> "Insurance)"
        sqldict['Name'].append(name)
        sqldict['CoType'].append(section)
        sqldict['License_Type'].append(category)
        sqldict['Address_1'].append(address)
        sqldict['Phone'].append(phone)
        sqldict['Fax'].append(fax)
        sqldict['Website'].append(website)
        sqldict['Email'].append(email)
        add_common(reg)


def add_common(reg):
    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegCtry'].append(reg.split()[0])
    sqldict['RegCode'].append(reg.split()[1])
    sqldict['ListCode'].append(reg.split()[2])
    sqldict['ListLabel'].append(listlabeldict[reg])
    sqldict['ListName'].append(Typology[reg])
    sqldict['RegulationType'].append('Regulated')
    bourange_same_length_array(sqldict)


parserdict = {
    regulatorName + ' 1': (parse_list1_banks,    extract_table_rows_ocr),
    regulatorName + ' 2': (parse_list2_mfi,      extract_table_rows_text),
    regulatorName + ' 4': (parse_list4_brokers,  extract_table_rows_text),
    regulatorName + ' 5': (parse_list5_pension,  extract_table_rows_ocr_borderless),
    regulatorName + ' 6': (parse_list6_payment,  extract_table_rows_text),
    regulatorName + ' 7': (parse_list7_insurers, extract_table_rows_text),
}

In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------
for k, reg in enumerate(parserdict):
    print(f"[INFO] : Working {k+1}/{len(parserdict)} _({reg})_ {Typology[reg]}")
    rows_before = len(sqldict['Name'])
    parser, extractor = parserdict[reg]
    pdf_path = download_list_pdf(reg)
    rows = extractor(pdf_path)
    parser(rows, reg)
    print(f"[INFO] : {reg} collected {len(sqldict['Name']) - rows_before} rows")

[INFO] : Working 1/6 _(RW NBRW 1)_ List of Licensed Banks


   downloaded 'List of Supervised Banks' -> RW_NBRW_1_LIST_OF_SUPERVISED_BANKS.pdf


[INFO] 2026-07-27 13:10:49,873 [RapidOCR] base.py:23: Using engine_name: onnxruntime


[INFO] 2026-07-27 13:10:49,929 [RapidOCR] download_file.py:60: File exists and is valid: /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/.venv/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx


[INFO] 2026-07-27 13:10:49,930 [RapidOCR] main.py:63: Using /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/.venv/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx


...

[INFO] 2026-07-27 13:10:53,981 [RapidOCR] main.py:63: Using /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/.venv/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_rec_small.onnx


   downloaded 'List of Pension service providers' -> RW_NBRW_5_LIST_OF_UPDATED_SERVICE_PROVIDERS.pdf
   [OCR] engine=RapidOCR, img2table=2.0.0


   [OCR] borderless_tables=True: 19 raw rows
   [OCR]    Type of service Providers | LICENSE/ACCREDITATION LETTER | Contacts
   [OCR]    CORPORATE TRUSTEE | License Number | 
   [OCR]    BPR Bank Rwanda Plc | No 001/2023 | (+250) 785640768
   [OCR]    ADMINISTRATORS | License Number/Accreditation | Contacts
   [OCR]    ZAMARA AAIB | ADM 01/2022 | (+250)788588132
   [OCR]    Liaison Financial Services R | ADM 01/2022 | (+250)250600430
   [OCR]    BK Capital Ltd | ADM 02/2022 | (+250) 788143141
   [OCR]    Axis Pensions Ltd | ADM 03/2022 | (+250) 785063930
   [OCR]    Maj (Rtd) KAYIGIRE Augustin | ADM 003/2021 | (+250)788303512
   [OCR]    NIYIBIZI Arnold | 2310/2021-02011BNR(803.4.20) | (+250)788505411
   [OCR]    INVESTMENT MANAGERS | License Number/Accreditation | Contacts
   [OCR]    BK Capital Ltd | Our/Ref:2310/2022-02350/0020 | (+250) 788143141
   [OCR]    Liaison Financial Services R | Our/Ref:2310/2021-01988/0020 | (+250)250600430
   [OCR]    Rwanda National Investment T | Our/R

   downloaded 'Licensed Institutions as of June 2026' -> RW_NBRW_6_Licensed_Institutions_-_June_2026.pdf
[INFO] : RW NBRW 6 collected 47 rows
[INFO] : Working 6/6 _(RW NBRW 7)_ List of Insurance Companies


   downloaded 'List of Insurance Companies' -> RW_NBRW_7_LIST_OF_LICENSED_INSURERS-UPDATED-May_2026.pdf
[INFO] : RW NBRW 7 collected 18 rows


In [7]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))

Saved 181 rows to /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/RW NBRW/RW NBRW SQL Ready 2026-07-27 13.10.47.xlsx
